# Lab — Exoplanet Populations

**Topic:** Pine Ch. 8 (Plotting), continued. Loading from Pine §5.3.  
**Files you need:** `exoplanet_data.csv`, `README.md` (in the same folder as this notebook)

Last week you worked with 10 bodies in our own Solar System. This week you have **5,487 confirmed exoplanets** discovered between 1995 and 2026 by every major detection method. The point of this lab is no longer to look at individual worlds. It's to look at *populations*: when you have thousands of planets, what patterns appear that you couldn't see with just nine?

Same structure as last week: each exercise has a 📖 **Reference** pointing back to Pine, and a short **Reflect** prompt at the end. The Reflect prompts are good fodder for your zine.

**One thing that's new:** real exoplanet data is *incomplete*. Different detection methods measure different things — some give you a planet's size, others give you its mass, almost none give you both at once. That means the `pl_rade`, `pl_bmasse`, and `pl_eqt` columns each have NaN values scattered through them. The README explains why. Exercise 2 will show you how to handle this with a Boolean mask (Pine §4.4.4 — same trick you used in the control flow assignment).

## Setup

Run this cell first. Same imports as last week (Pine §8.1).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Exercise 1 — Load it and look at it

**Goal:** get the data into Python and confirm you can pull pieces out of it.

The CSV has three text columns at the front (`pl_name`, `hostname`, `discoverymethod`) and seven numeric columns after. That means we'll need **two** calls to `np.loadtxt`: one to read the text columns as strings, and one to read the numbers. The README has the column order.

1. Load the text columns into separate arrays using `np.loadtxt` with `dtype=str`, `delimiter=','`, `skiprows=1`, and `usecols=(0, 1, 2)`. Then `unpack=True` to get three arrays out.
2. Load the numeric columns the same way you did last week — `unpack=True` plus `usecols=range(3, 10)` to skip the three text columns.
3. Assign descriptive names to each column. There are 7 numeric ones: `disc_year`, `period`, `radius`, `mass`, `Teq`, `Teff_star`, `dist`.
4. Print:
   - the total number of planets in the dataset
   - the name and orbital period of the planet with the **shortest year** (smallest `period`)
   - the name and radius of the **largest** planet by radius (be careful — `np.argmax` doesn't ignore NaN; use `np.nanargmax` instead)

*Hint:* `np.nanargmax(radius)` returns the **index** of the largest non-NaN value. Feed that index into `pl_name` to get the name. Same trick as last week, just NaN-aware.

**Reference:** Pine §5.3.2 for `np.loadtxt`. `np.nanargmax` / `np.nanargmin` are NaN-aware versions of the functions you used last week (Pine introduces them in §9.5 with `np.nanmean`, same pattern).

In [ ]:
# Load the three text columns
pl_name, hostname, method = np.loadtxt('exoplanet_data.csv',
                                       dtype=str, delimiter=',',
                                       skiprows=1, usecols=(0, 1, 2),
                                       unpack=True)

# Load the numeric columns
(disc_year, period, radius, mass,
 Teq, Teff_star, dist) = np.loadtxt('exoplanet_data.csv',
                                    delimiter=',', skiprows=1,
                                    usecols=range(3, 10), unpack=True)

print(f'Number of planets: {len(pl_name)}')

## Exercise 2 — The mass-radius diagram, with NaN handling

This is one of the most famous plots in exoplanet science. Each planet is a point at (mass, radius); the pattern reveals what kinds of planets are common, where the bulk of detections cluster, and roughly how dense planets in different size ranges are.

**But:** many planets have a measured mass but no measured radius (radial velocity detections), or a measured radius but no measured mass (some transit detections). For *this* plot we need both, so we have to filter out the NaN values first.

1. Build a Boolean mask that's `True` only for planets where **both** `mass` and `radius` are non-NaN. The pattern is:
   ```python
   mask = ~np.isnan(mass) & ~np.isnan(radius)
   ```
   `np.isnan(x)` returns `True` for NaN entries; `~` flips True↔False. The `&` is element-wise AND on Boolean arrays (Pine §4.4.4).
2. Make a **log-log** scatter plot of `mass[mask]` vs. `radius[mask]`. Use small blue dots — `'.C0'` works well with this many points (no marker outline at this scale).
3. Add Earth and Jupiter as reference points. Hardcode their values:
   - Earth: mass = 1, radius = 1
   - Jupiter: mass ≈ 318, radius ≈ 11.2
   Plot each with a big colored star — `'*C3'` for red stars — and label them using `plt.text`.
4. Add axis labels with units, a title, and `plt.tight_layout()`.
5. Print how many planets are in the plot (i.e., `mask.sum()`) — this is the subset that has both measurements.

**Reference:** Pine §4.4.4 (Boolean indexing — the masking pattern), §8.3.2 (`plt.loglog`), §8.2.1 (format strings and `markersize`), §8.7 (`plt.text`).

In [ ]:
# TODO: build the mask, count how many planets it picks out


plt.figure(1, figsize=(7, 5))

# TODO: plt.loglog the masked mass and radius


# TODO: add Earth and Jupiter as red stars, label with plt.text


plt.xlabel('Planet mass (Earth masses)')
plt.ylabel('Planet radius (Earth radii)')
plt.title('Exoplanet mass-radius diagram')
plt.tight_layout()
plt.show()

**Reflect:** Where does the dense cluster of exoplanets sit relative to Earth and Jupiter? What kind of planet do we find the most of? (You may want to look up the terms *super-Earth* and *sub-Neptune* — both fall in the gap between the two reference points.) Also: of the 5,487 planets in the file, how many did you actually plot? Why so few?

## Exercise 3 — Hot or cold? Period vs. equilibrium temperature

There's a physical reason planets close to their star are hot: they absorb more of the star's light. Let's see if the data show this.

1. Build a Boolean mask for planets with non-NaN `Teq` (same `~np.isnan(...)` pattern as Exercise 2).
2. Make a **semi-log** plot of orbital period (x-axis, log scale, in days) vs. equilibrium temperature (y-axis, linear, in K). Use small orange dots (`'.C1'`).
3. Add a horizontal line at 273 K (water freezes) and another at 373 K (water boils) using `plt.axhline`. Use a light-gray dashed style — Pine uses `color='gray', linestyle='--'`. These mark the temperature range where liquid water could exist on the surface, ignoring atmospheric effects.
4. Label the axes and give the plot a title.

**Reference:** Pine §8.3.1 (`plt.semilogx`). `plt.axhline` is on p. 170 of the basic plotting section, just before Table 8.1.

In [ ]:
# TODO: build a mask for non-NaN Teq


plt.figure(2, figsize=(7, 5))

# TODO: semi-log scatter of period vs. Teq for masked planets


# TODO: add the liquid-water reference lines at 273 K and 373 K


plt.xlabel('Orbital period (days)')
plt.ylabel('Equilibrium temperature (K)')
plt.title('Period vs. equilibrium temperature')
plt.tight_layout()
plt.show()

**Reflect:** Does the trend match the physical intuition (shorter period → closer to star → hotter)? Where does the bulk of the dataset live on this plot? Note that you're only seeing planets with a measured Teq — about 70% of the catalog drops out here. What kinds of planets do you think are missing from the plot, and why might that be?

## Exercise 4 — Discovery methods over time

The catalog of known exoplanets grew enormously between 1995 and today. But not all the growth came from the same place: different detection methods dominated at different times. Let's see this directly.

We'll focus on the three methods with the most planets: **Transit** (4533), **Radial Velocity** (899), and **Transit Timing Variations** (27).

1. For each method, build a Boolean mask: `m_transit = (method == 'Transit')`, and similarly for the other two.
2. For each method, compute the cumulative count over time:
   ```python
   years_t, counts_t = np.unique(disc_year[m_transit], return_counts=True)
   cumul_t = np.cumsum(counts_t)
   ```
3. Plot all three on the same axes using **three separate `plt.plot` calls**, each with a different color and a `label=` argument. Use connected-line-with-markers style like `'-oC0'`, `'-sC1'`, `'-^C2'` (different markers help if the lines get tangled).
4. Call `plt.legend()` to display the legend.
5. Label axes and add a title.

**Reference:** Pine §8.2 walks through multiple plots with a legend — see the `wavy_pulse` example. `np.unique` and `np.cumsum` are Pine Ch. 5. Boolean masks are §4.4.4.

In [ ]:
# TODO: build masks for the three top methods


# TODO: for each method, get (years, cumulative counts) using np.unique + np.cumsum


plt.figure(3, figsize=(8, 5))
# TODO: three plt.plot calls, one per method, each with a label


plt.legend()
plt.xlabel('Discovery year')
plt.ylabel('Cumulative number of planets')
plt.title('Exoplanet discoveries over time, by method')
plt.tight_layout()
plt.show()

**Reflect:** When did the transit method overtake radial velocity? What space mission was responsible for the dramatic jump? (Hint: it launched in 2009 and finished its primary mission in 2013. The follow-up to it launched in 2018.) Why do you think radial velocity grows much more steadily than transit — what's different about how the two methods work?

## Exercise 5 — The temperate zone (Optional)

Of these 5,487 planets, which ones could in principle have liquid water on their surface? In Exercise 3 you drew a rough band at 273–373 K. Let's pick those planets out and look at them.

1. Build a Boolean mask for planets with `Teq` between 273 and 373 K. Combine it with a non-NaN check using `&`:
   ```python
   temperate = ~np.isnan(Teq) & (Teq >= 273) & (Teq <= 373)
   ```
2. Print how many you found, then print the first 15 of them: name, radius, mass, equilibrium temperature, and discovery method. (Some will have NaN for radius or mass — that's fine, just print whatever's there.)
3. Make a final plot: orbital period (x-axis, **log** scale) vs. equilibrium temperature (y-axis), with **all** Teq-measured planets shown as small gray dots in the background, and the temperate planets highlighted in a bright color (`'oC3'` red, larger). Add a horizontal band shading the liquid-water region using `plt.axhspan(273, 373, color='C0', alpha=0.15)`.

**Reference:** Pine §4.4.4 (Boolean indexing). `plt.axhspan` is a close cousin of `plt.axhline` from §8.2.

In [ ]:
# TODO: build the temperate mask


# TODO: print the count, then list the first 15 temperate planets
# (a for loop with f-strings reads nicely; Pine §6 covers for loops)



In [ ]:
plt.figure(4, figsize=(8, 5))

# Background: all planets with a measured Teq (use the mask from Ex. 3)
# TODO: plot the background in light gray with small '.' markers


# Foreground: the temperate planets in red
# TODO: plt.semilogx the temperate planets


# Shade the liquid-water band
plt.axhspan(273, 373, color='C0', alpha=0.15)

plt.xlabel('Orbital period (days)')
plt.ylabel('Equilibrium temperature (K)')
plt.title('Where are the temperate exoplanets?')
plt.tight_layout()
plt.show()